In [0]:
batch_path = (
    "s3://stock-market-pipeline-mule-2026/"
    "raw/batch/SP500_Historical_Data.csv"
)

batch_df = (
    spark.read
    .option("header", "true")
    .csv(batch_path)
)

display(batch_df.limit(10))

Ticker,Date,Open,High,Low,Close,Adj Close,Volume
A,2000-01-03,47.07,47.18,40.27,43.04,43.04,4674353
A,2000-01-04,40.72,41.17,38.7,39.75,39.75,4765083
A,2000-01-05,39.6,39.75,36.05,37.28,37.28,5758642
A,2000-01-06,36.83,37.06,34.74,35.86,35.86,2534434
A,2000-01-07,35.3,39.41,35.27,38.85,38.85,2819626
A,2000-01-10,41.24,41.62,40.38,41.21,41.21,2148446
A,2000-01-11,41.21,41.21,39.71,40.65,40.65,1855985
A,2000-01-12,40.65,40.65,38.29,39.82,39.82,1429874
A,2000-01-13,40.87,41.73,39.45,40.42,40.42,1134337
A,2000-01-14,40.05,41.47,40.05,40.87,40.87,1316916


In [0]:
from datetime import datetime
from math import isfinite

from pyspark.sql import functions as F
from pyspark.sql.types import StringType


def validate_stock_row(
    ticker,
    date_value,
    open_price,
    high_price,
    low_price,
    close_price,
    adjusted_close,
    volume,
):
    required_values = [
        ticker,
        date_value,
        open_price,
        high_price,
        low_price,
        close_price,
        adjusted_close,
        volume,
    ]

    if any(value is None or str(value).strip() == "" for value in required_values):
        return "missing_value"

    try:
        datetime.strptime(str(date_value), "%Y-%m-%d")
    except ValueError:
        return "invalid_date"

    try:
        open_price = float(open_price)
        high_price = float(high_price)
        low_price = float(low_price)
        close_price = float(close_price)
        adjusted_close = float(adjusted_close)
        volume = int(volume)
    except ValueError:
        return "invalid_number"

    prices = [
        open_price,
        high_price,
        low_price,
        close_price,
        adjusted_close,
    ]

    if not all(isfinite(price) and price > 0 for price in prices):
        return "invalid_price"

    if volume < 0:
        return "negative_volume"

    if high_price < max(open_price, low_price, close_price):
        return "invalid_high_price"

    if low_price > min(open_price, high_price, close_price):
        return "invalid_low_price"

    return None


validate_stock_udf = F.udf(validate_stock_row, StringType())

validated_df = batch_df.withColumn(
    "validation_error",
    validate_stock_udf(
        F.col("Ticker"),
        F.col("Date"),
        F.col("Open"),
        F.col("High"),
        F.col("Low"),
        F.col("Close"),
        F.col("Adj Close"),
        F.col("Volume"),
    ),
)

valid_count = validated_df.filter(
    F.col("validation_error").isNull()
).count()

invalid_count = validated_df.filter(
    F.col("validation_error").isNotNull()
).count()

print(f"Valid records: {valid_count:,}")
print(f"Invalid records: {invalid_count:,}")

display(
    validated_df
    .filter(F.col("validation_error").isNotNull())
    .limit(10)
)

Valid records: 2,703,530
Invalid records: 1


Ticker,Date,Open,High,Low,Close,Adj Close,Volume,validation_error
HUBB,2021-05-05,181.34,183.8,181.55,183.33,183.33,127234,invalid_low_price


In [0]:
processed_batch_path = (
    "s3://stock-market-pipeline-mule-2026/"
    "processed/batch/stock_prices/"
)

quarantine_batch_path = (
    "s3://stock-market-pipeline-mule-2026/"
    "quarantine/batch/invalid_stock_prices/"
)

cleaned_batch_df = (
    validated_df
    .filter(F.col("validation_error").isNull())
    .select(
        F.upper(F.trim(F.col("Ticker"))).alias("ticker"),
        F.to_date(F.col("Date"), "yyyy-MM-dd").alias("trade_date"),
        F.col("Open").cast("double").alias("open_price"),
        F.col("High").cast("double").alias("high_price"),
        F.col("Low").cast("double").alias("low_price"),
        F.col("Close").cast("double").alias("close_price"),
        F.col("Adj Close").cast("double").alias("adjusted_close_price"),
        F.col("Volume").cast("long").alias("volume"),
        F.lit("kaggle").alias("source"),
        F.current_timestamp().alias("processed_at"),
    )
)

invalid_batch_df = (
    validated_df
    .filter(F.col("validation_error").isNotNull())
    .withColumn("quarantined_at", F.current_timestamp())
)

cleaned_batch_df.write.mode("overwrite").parquet(processed_batch_path)

invalid_batch_df.write.mode("overwrite").parquet(quarantine_batch_path)

print("Valid records saved as Parquet:")
print(processed_batch_path)

Valid records saved as Parquet:
s3://stock-market-pipeline-mule-2026/processed/batch/stock_prices/
